In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/pragati8979saini@gmail.com/FMCG-Project-using-databricks/consolidated_pipeline/setup_1/utilities


In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source","customers","Data Source")


In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = f's3://sportsbar-cult/{data_source}/*csv'
print(catalog, data_source)
print(base_path)


In [0]:
df= spark.read.format("csv").option("header", "true").load(base_path)
display(df.limit(5))


In [0]:
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(base_path)
    .withColumn("read_timestamp", f.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)
display(df.limit(5))

In [0]:
df.printSchema()

In [0]:
df.write\
    .mode("overwrite")\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")  

**SILVER PROCESSING**

In [0]:
df_bronze= spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
display(df_bronze.limit(5))

Silver transformation begins

In [0]:
df_bronze.printSchema()

Remove duplicates

In [0]:
df_duplicates=df_bronze.groupBy("customer_id").count().filter("count>1")
display(df_duplicates)

In [0]:
print("Rows before duplicates dropped: ",df_bronze.count())
df_silver=df_bronze.dropDuplicates(["customer_id"])
print("Rows after duplicates dropped: ",df_silver.count())


remove eading white space

In [0]:
#checking wchich customer name has leading spaces
#df_silver.filter(df_silver.customer_name.startswith(" ")).show()
#trim : is only used to remove white spaces from the beginning and end of a string, it doen't remove spaces in between

display(df_silver.filter(f.col("customer_name")!= f.trim(f.col("customer_name"))))

In [0]:
df_silver= df_silver.withColumn("customer_name",f.trim(f.col("customer_name")))

In [0]:
df_silver.select("city").distinct().show()

In [0]:
#remove dirty data in city column
#typos -> correct names
city_mapping = {
    "Bengaluru" : 'Bengaluru',
    "Bangalore" : 'Bengaluru',
    "Banglore" : 'Bengaluru',

    "Delhi" : 'Delhi',
    "New Delhi" : 'Delhi',

    "Hyderabad" : 'Hyderabad',
    "Hyderabadd": 'Hyderabad'
}

allowed = ["Bengaluru","Delhi","Hyderabad"]

#replace: df.replace(dict,subset)
#replace(to_replace: Union[LiteralType, List[LiteralType], Dict[LiteralType, OptionalPrimitiveType]], value, subset: Optional[List[str]])

df_silver= (
    df_silver
    .replace(city_mapping,subset=["city"])
    .withColumn(
        "city",
        f.when(f.col("city").isNull(),None)
        .when(f.col("city").isin(allowed),f.col("city"))
        .otherwise(None)
    )
    )

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
#to fix upper inner case using initcap: Title case fix
df_silver= (
  df_silver.withColumn(
    "customer_name",
    f.when(f.col("customer_name").isNull(),None)
     .otherwise(f.initcap("customer_name"))                     
    )
)

df_silver.select("customer_name").distinct().show()

In [0]:
df_silver.select('*').show()
# here we can see there are city having null values so for that we need to check with business which value need to hardcore in them

df_silver.filter(f.col("city").isNull()).show(truncate=False)


In [0]:
#will create array to find all records having the record with the same customer_name
null_customer_names=['Sprintx Nutrition','Zenathlete Foods','Peak Performance Store','Primefuel Nutrition','Recovery Lane','Endurance Foods']

df_silver.filter(f.col('customer_name').isin(null_customer_names)).show(truncate=False)

In [0]:
#Business confirmation notes: city correction confirmed by business
customer_city_fix = {
    789101: 'Bengaluru',
    789403: 'Bengaluru',
    789420: 'Bengaluru',
    789421: 'Bengaluru',
    789422: 'Bengaluru',
    789503: 'Bengaluru',
    789520: 'Delhi',
    789521: 'Delhi',
    789522: 'Delhi',
    789603: 'Delhi'
}

df_fix= spark.createDataFrame(customer_city_fix.items(),['customer_id','fix_city'])
df_fix.show()

In [0]:
#join the datframe df_fix into df_silver to remove null city
df_silver= (
     df_silver
            .join(df_fix, on='customer_id', how='left')
            .withColumn(
                "city",
                f.when(f.col("city").isNull(),f.col("fix_city"))
                .otherwise(f.col("city"))
                # instead of using when and otherwise:
                #f.coalesce(f.col("city"), f.col("fix_city")) — this says "Keep the city if it's there; if not, take the fix."
            )
            .drop("fix_city")
)
display(df_silver)

In [0]:
#now there iss no null city in city column
df_silver.filter(f.col("city").isNull()).show(truncate=False)


In [0]:
#Since at gold layer customerid is string so will be converting the datatype for customer_id column in silver datframe as well
df_silver= df_silver.withColumn("customer_id",f.col("customer_id").cast("string"))
df_silver.printSchema()


In [0]:
# now will try to create the same schema like present in gold layer
#customer_code->customer_id
#customer->customer_name-city
#will create new column market and set the default value as India
#new column platform and channel and set the default value as 

df_silver=(
        df_silver.withColumn(
        "customer",
        #concat_ws stands for concatenation with seprator
        f.concat_ws("-",f.col("customer_name"),f.coalesce(f.col("city"),f.lit("Unknown"))
        )
        )
        .withColumn(
            "market",
            f.lit("India")
        )
        .withColumn(
            "platform",
            f.lit("Sports Bar")
        )
        .withColumn(
            "channel",
            f.lit("Acquisition")
        )
       .withColumnRenamed("customer_id","customer_code")
       .drop("customer_name","city")
)
display(df_silver)

In [0]:
#will write the dataframe to delta table
df_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "True")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")


**GOLD** **PROCESSING**

In [0]:
df_gold=df_silver.select("customer_code","customer","market","platform","channel")
df_gold.printSchema()
display(df_gold)


In [0]:
#write into gold delta table
df_gold.write\
    .mode("overwrite")\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "True")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
#merge the gold data and sports bar gold data into gold data
#Deltatable.forName: ACID operation, can perform sql via alias(), can perform history,merge
delta_table_gold= DeltaTable.forName(spark,"fmcg.gold.dim_customers")
#spark.table: its a dataframe and perform analytical and transformation operation
df_child_customer= spark.table("fmcg.gold.sb_dim_customers")

In [0]:
delta_table_gold.alias("target").merge(
    source=df_child_customer.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
#display(delta_table_gold.history())
row_count = delta_table_gold.toDF().count()
print(row_count)